# Chat Model 常用参数

上一节学习了 `init_chat_model()` 的基本用法。本节详细介绍调用模型时常用的控制参数。

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

print("导入完成")

导入完成


## temperature——控制创造性与确定性

temperature 是最常用的参数，取值范围 0 到 2，控制模型输出的随机程度。

In [5]:
# 同一问题，不同 temperature 的对比
question = "用一句话形容拜登"
question = "请严格输出一句话：'苹果是红色的' 的英文翻译，不要有任何多余字符。"

# temperature=0：输出非常确定
model_low = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
resp1 = model_low.invoke(question)
resp2 = model_low.invoke(question)
print(f"temperature=0 第1次: {resp1.content}")
print(f"temperature=0 第2次: {resp2.content}")
print(f"两次结果相同: {resp1.content == resp2.content}")

# temperature=1.5：输出多样化
model_high = init_chat_model("deepseek:deepseek-v4-flash", temperature=1.5)
resp1 = model_high.invoke(question)
resp2 = model_high.invoke(question)
print(f"temperature=1.5 第1次: {resp1.content}")
print(f"temperature=1.5 第2次: {resp2.content}")

print("模型已创建，取消注释 invoke 调用即可测试")

temperature=0 第1次: Apples are red.
temperature=0 第2次: The apple is red.
两次结果相同: False
temperature=1.5 第1次: Apples are red.
temperature=1.5 第2次: Apples are red.
模型已创建，取消注释 invoke 调用即可测试


In [8]:
import time

# ===============================
# 【低温测试 temp=0】稳定如老狗
# ===============================
def get_low_prompt():
    return f"请填空：1+1=?，只需输出阿拉伯数字。防缓存码：{time.time()}"

model_low = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
resp1_low = model_low.invoke(get_low_prompt())
resp2_low = model_low.invoke(get_low_prompt())

print(f"【低温测试 temp=0】")
print(f"第1次: {resp1_low.content}")
print(f"第2次: {resp2_low.content}")
print(f"两次是否相同: {resp1_low.content == resp2_low.content}\n")


# ===============================
# 【高温测试 temp=1.5】群魔乱舞
# ===============================
def get_high_prompt():
    # 用极其开放的提示词，逼迫模型发散
    return f"请随机想一种包含两个字的动物名字（不要解释，只输出动物名）。防缓存码：{time.time()}"

# 注意这里：top_p 直接作为参数传入，不再放进 model_kwargs
model_high = init_chat_model(
    "deepseek:deepseek-v4-flash", 
    temperature=1.5, 
    top_p=1.0  # 显式传入，消除 Warning，释放 100% 的词库
)

resp1_high = model_high.invoke(get_high_prompt())
resp2_high = model_high.invoke(get_high_prompt())

print(f"【高温测试 temp=1.5】")
print(f"第1次: {resp1_high.content}")
print(f"第2次: {resp2_high.content}")
print(f"两次是否相同: {resp1_high.content == resp2_high.content}")

【低温测试 temp=0】
第1次: 2
第2次: 2
两次是否相同: True

【高温测试 temp=1.5】
第1次: 老虎
第2次: 兔子
两次是否相同: False


| temperature 值 | 效果 | 适用场景 |
| --- | --- | --- |
| 0 ~ 0.3 | 输出稳定、确定，每次结果几乎一致 | 数据提取、分类、代码生成、翻译 |
| 0.5 ~ 0.7 | 适度的创造性，输出自然但不偏离主题 | 日常对话、内容总结 |
| 0.8 ~ 1.2 | 输出多样化，有较多发挥空间 | 创意写作、头脑风暴 |
| 1.3 ~ 2.0 | 输出非常随机，可能出现意外内容 | 探索性生成（不太推荐用于生产） |

> `temperature=0` 不等于"完全相同"。由于浮点精度差异，极端情况下仍可能有微小差异。需要绝对确定性时，有些模型提供了 `seed` 参数。

## max_tokens——控制输出长度与成本

max_tokens 限制模型输出的最大 Token 数。一个 Token 大约相当于 0.75 个英文单词或 0.5 个中文字。

In [18]:
model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)

# 限制 30 个 Token：使用 .bind() 绑定参数，然后再 invoke
response_short = model.bind(max_tokens=30).invoke("详细介绍一下菜鸟教程 RUNOOB 平台")

print(f"限制 30 tokens ({len(response_short.content)} 字符):")
print(response_short.content)
print("-" * 30)

# 限制 200 个 Token
response_long = model.bind(max_tokens=200).invoke("详细介绍一下菜鸟教程 RUNOOB 平台")

print(f"限制 200 tokens ({len(response_long.content)} 字符):")
print(response_long.content)

限制 30 tokens (0 字符):

------------------------------
限制 200 tokens (10 字符):
我来为你详细介绍一下


> `max_tokens` 是对输出长度的硬限制。如果设置过低，回答可能在句中突然截断。一般建议："一句话回答"类场景 30~100，"详细解释"类场景 500~2000。

## timeout 与 max_retries——网络可靠性

生产环境中网络请求可能失败。这两个参数控制请求行为。

In [ ]:
# 生产环境推荐配置
model = init_chat_model(
    "deepseek:deepseek-v4-flash",
    timeout=30,       # 单次请求最多等待 30 秒
    max_retries=3,    # 失败后最多重试 3 次
)

try:
    response = model.invoke("菜鸟教程 RUNOOB 是什么？")
    print(f"调用成功: {response.content[:50]}...")
except Exception as e:
    print(f"调用失败: {e}")

调用成功: “菜鸟教程”和“RUNOOB”其实指的是**同一个网站**（runoob.com）。

简单来说，*...
模型已创建，取消注释 invoke 调用即可测试


| 参数 | 说明 | 建议值 |
| --- | --- | --- |
| timeout | 单次请求最大等待时间（秒）。None 表示不限制 | 30~60 |
| max_retries | 失败后重试次数。0 表示不重试 | 2~3 |

设置 `timeout=30, max_retries=3` 的费时分析：
```
单次成功: ~2s
第1次失败+重试成功: ~2s + ~2s = ~4s
全部失败: 4次 × 30s = 最多 120s 才抛出异常
```

## base_url——自定义 API 地址

通过代理、中转服务或私有部署访问模型时使用。

In [ ]:
# 场景 1：通过代理访问 OpenAI
# model = init_chat_model(
#     "deepseek:deepseek-v4-flash",
#     base_url="https://your-proxy-domain.com/v1",
# )

# 场景 2：兼容 OpenAI 接口的第三方服务
# model = init_chat_model(
#     "deepseek:deepseek-v4-flash",
#     base_url="https://api.third-party.com/v1",
#     api_key="your-third-party-key",
# )

# 场景 3：连接本地模型 (vLLM/Ollama)
# model = init_chat_model(
#     "openai:qwen2.5",
#     base_url="http://localhost:8000/v1",
#     api_key="not-needed",
# )

print("base_url 参数支持代理、中转、本地部署等场景")

> `base_url` 改变的是 API 端点地址，但 `provider` 参数决定了行为模式。例如 `provider="openai"` 使用 OpenAI 消息格式，即使 `base_url` 指向别的服务。确保目标服务兼容你指定的 provider 格式。

## 其他常用参数

### top_p——核采样

模型只会从累积概率达到 `top_p` 的词中采样。`top_p=0.1` 表示只从最高概率 10% 的词中选择。

In [ ]:
model = init_chat_model("deepseek:deepseek-v4-flash", top_p=0.9)
response = model.invoke("介绍菜鸟教程 RUNOOB")
print(response.content[:100])

菜鸟教程（RUNOOB），其官方域名为 `www.runoob.com`，是一个非常知名且广受欢迎的**中文编程技术学习网站**，尤其对**编程初学者**极为友好。

以下是关于它的详细介绍：

##
模型已创建，取消注释 invoke 调用即可测试


> 建议只调整 `temperature` 或 `top_p` 中的一个，不要同时调节。同时设置时行为可能难以预测。

### stop——停止序列

模型遇到停止序列中的词时会立即停止生成。

In [21]:
model = init_chat_model("deepseek:deepseek-v4-flash")

# stop 参数：遇到换行就停止，只返回第一个结果
response = model.invoke(
    "列出五个编程学习网站，每个一行",
    stop=["\n"]
)
print(f"限制 stop=['\\n']: {response.content}")

限制 stop=['\n']: Codecademy  


### seed——可重复性（部分模型支持）

相同的 seed + 相同的输入 = 相同的输出。

In [22]:
# 某些模型支持 seed 参数
model = init_chat_model("deepseek:deepseek-v4-flash", seed=42, temperature=0)
resp1 = model.invoke("介绍菜鸟教程")
resp2 = model.invoke("介绍菜鸟教程")
print(f"seed=42, 结果相同: {resp1.content == resp2.content}")

seed=42, 结果相同: False


## 参数速查表

| 参数 | 类型 | 默认值 | 何时使用 |
| --- | --- | --- | --- |
| temperature | float | 因模型而异 | 任务需要稳定性时设为 0~0.3，需要创造性时设为 0.7~1.0 |
| max_tokens | int | 模型上限 | 输出长度需要控制时 |
| timeout | int/float | None | 生产环境建议始终设置 |
| max_retries | int | 因模型而异 | 网络不稳定时建议 2~3 |
| base_url | str | 官方地址 | 使用代理、中转或本地服务时 |
| top_p | float | 1.0 | 需要核采样控制时（替代 temperature） |
| stop | list[str] | 无 | 需要精确控制输出结尾时 |

**术语：**

- **temperature**：控制输出随机性的参数（0=确定性，1=创造性）
- **max_tokens**：限制输出最大 Token 数
- **top_p**：核采样参数，从累积概率达到 top_p 的词中采样
- **stop**：停止序列，模型遇到这些词时停止生成
- **seed**：随机种子，相同 seed 产生相同结果